Celda 1 — Instalación de dependencias

In [ ]:
!pip install -q anthropic tqdm pandas

Celda 2 — Subir el archivo y configurar la API key

In [ ]:
from google.colab import files
uploaded = files.upload()  # selecciona sentiment_ground_truth.csv

In [ ]:
import getpass
import os

os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Pega tu ANTHROPIC_API_KEY: ")

Celda 3 — Configuración

In [ ]:
GROUND_TRUTH_PATH = "sentiment_ground_truth.csv"
OUTPUT_PATH = "predicciones_sentimiento.json"
CHECKPOINT_PATH = "predicciones_sentimiento_checkpoint.json"

MODEL = "claude-sonnet-5"   # comparativa de referencia frente a Gemma (gratuito);
                             # ~$2/$10 por millón de tokens in/out (precio de lanzamiento hasta el 31/08/2026)
MAX_TOKENS = 200

CHECKPOINT_EVERY = 25      # guarda a disco cada 25 filas por si Colab se desconecta
MAX_RETRIES = 5
BASE_BACKOFF_SECONDS = 2

Celda 4 — Cliente de Anthropic y prompt del sistema

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

SYSTEM_PROMPT = """Eres un analista financiero senior, experto en interpretar el impacto de noticias en los mercados.
Tu tarea es clasificar el sentimiento de mercado de una noticia para un activo/ticker concreto.

Responde ÚNICAMENTE con un objeto JSON, sin texto adicional antes ni después, con este formato exacto:
{"sentimiento": "bullish|bearish|sideways", "confianza": 0.0}

Reglas:
- "sentimiento" debe ser exactamente una de estas tres palabras: bullish, bearish, sideways.
- "confianza" es tu propia estimación (0.0 a 1.0) de cuán seguro estás de esa clasificación.
- No incluyas explicaciones, razonamiento ni texto fuera del JSON."""

Celda 5 — Construcción del prompt y parseo de la respuesta

In [ ]:
import re
import json

VALID_LABELS = {"bullish", "bearish", "sideways"}

# Normaliza también las palabras clave en español/inglés a la etiqueta canónica
KEYWORD_TO_LABEL = {
    "bullish": "bullish", "alcista": "bullish",
    "bearish": "bearish", "bajista": "bearish",
    "sideways": "sideways", "lateral": "sideways",
}

def build_user_prompt(row):
    return f"""Ticker: {row['ticker']}
Sector/Índice: {row['indice_sector']}
Fecha: {row['fecha']}
Titular de la noticia: "{row['titulo']}"

Clasifica el sentimiento de mercado de esta noticia para {row['ticker']}."""


def parse_response(raw_text):
    """Intenta parsear el JSON de la respuesta. Si falla, hace un fallback por palabras clave
    para no perder la fila por completo, y deja rastro en 'respuesta_cruda' para depurar.
    Solo se acepta como válida una de las tres etiquetas exactas (VALID_LABELS); cualquier
    otro valor se descarta (se devuelve None) para no contaminar las métricas con ruido."""
    try:
        cleaned = raw_text.strip()
        cleaned = re.sub(r"^```(json)?|```$", "", cleaned, flags=re.MULTILINE).strip()
        data = json.loads(cleaned)
        sentimiento = str(data.get("sentimiento", "")).strip().lower()
        confianza = data.get("confianza", None)
        confianza = float(confianza) if confianza is not None else None
        if sentimiento in VALID_LABELS:
            return sentimiento, confianza
        # JSON válido pero con una etiqueta no reconocida: no la aceptamos como buena,
        # caemos al fallback de palabras clave sobre el texto crudo
    except (json.JSONDecodeError, ValueError, AttributeError):
        pass

    lower = raw_text.lower()
    for kw, label in KEYWORD_TO_LABEL.items():
        if kw in lower:
            return label, None
    return None, None

Celda 6 — Llamada al LLM con exponential backoff

In [ ]:
import time
import random

# Solo tiene sentido reintentar errores transitorios: rate limit, sobrecarga (529)
# o problemas de servidor (5xx). Un 4xx (API key inválida, request mal formada, modelo
# inexistente, etc.) no se arregla reintentando, así que falla inmediatamente con el
# error original en vez de agotar los 5 reintentos y ocultar la causa real.
RETRYABLE_STATUS_CODES = {500, 502, 503, 529}

def call_llm_with_backoff(user_prompt):
    last_error = None
    for attempt in range(MAX_RETRIES):
        try:
            response = client.messages.create(
                model=MODEL,
                max_tokens=MAX_TOKENS,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": user_prompt}],
            )
            return response.content[0].text
        except anthropic.RateLimitError as e:
            last_error = e
            wait = BASE_BACKOFF_SECONDS * (2 ** attempt) + random.uniform(0, 1)
            print(f"  Rate limit (429). Reintentando en {wait:.1f}s...")
            time.sleep(wait)
        except anthropic.APIStatusError as e:
            status = getattr(e, "status_code", None)
            if status in RETRYABLE_STATUS_CODES:
                last_error = e
                wait = BASE_BACKOFF_SECONDS * (2 ** attempt) + random.uniform(0, 1)
                print(f"  Error de servidor ({status}). Reintentando en {wait:.1f}s...")
                time.sleep(wait)
            else:
                raise RuntimeError(
                    f"Error no recuperable de la API (status={status}): {e}"
                ) from e
        except anthropic.APIConnectionError as e:
            last_error = e
            wait = BASE_BACKOFF_SECONDS * (2 ** attempt) + random.uniform(0, 1)
            print(f"  Error de conexión. Reintentando en {wait:.1f}s...")
            time.sleep(wait)
    raise RuntimeError(
        f"Fallaron los {MAX_RETRIES} reintentos para esta fila. Último error: {last_error}"
    )

Celda 7 — Checkpointing (para no perder progreso si Colab se desconecta)

In [ ]:
import os

def load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, encoding="utf-8") as f:
            data = json.load(f)
        ids_procesados = {r["id"] for r in data}
        print(f"Checkpoint encontrado: {len(data)} filas ya procesadas, se reanuda desde ahí.")
        return data, ids_procesados
    return [], set()

def save_checkpoint(resultados):
    with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
        json.dump(resultados, f, ensure_ascii=False, indent=2)

Celda 8 — Bucle principal con tqdm

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

df = pd.read_csv(GROUND_TRUTH_PATH)
resultados, ids_procesados = load_checkpoint()

pendientes = df[~df["id"].isin(ids_procesados)]
print(f"Total filas: {len(df)} | Ya procesadas: {len(ids_procesados)} | Pendientes: {len(pendientes)}")

for i, (_, row) in enumerate(tqdm(pendientes.iterrows(), total=len(pendientes), desc="Clasificando sentimiento")):
    user_prompt = build_user_prompt(row)
    try:
        raw_text = call_llm_with_backoff(user_prompt)
        sentimiento, confianza = parse_response(raw_text)
    except RuntimeError as e:
        print(f"  ⚠️ Fila id={row['id']} falló tras reintentos: {e}")
        raw_text, sentimiento, confianza = None, None, None

    resultados.append({
        "id": int(row["id"]),
        "sentimiento_predicho": sentimiento,
        "confianza": confianza,
        "respuesta_cruda": raw_text,
    })

    if (i + 1) % CHECKPOINT_EVERY == 0:
        save_checkpoint(resultados)

save_checkpoint(resultados)
print(f"\nCompletado. {len(resultados)} predicciones procesadas.")

Celda 9 — Guardar y descargar el archivo final

In [ ]:
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

sin_etiqueta = sum(1 for r in resultados if r["sentimiento_predicho"] is None)
if sin_etiqueta:
    print(f"⚠️ {sin_etiqueta} filas quedaron sin etiqueta reconocida (revisa 'respuesta_cruda').")

from google.colab import files as colab_files
colab_files.download(OUTPUT_PATH)